# 📋 INSTRUCTOR NOTEBOOK — Lag-Embedded Feature Matrices

**Module 0 · Lesson 6 of 13 · Instructor edition**  
**Estimated class time:** 85–100 minutes  
**Source sequence:** Original Day 2  

**Prerequisite:** Lessons 4–5  

> **Do not distribute to students.** This edition contains solutions, teaching notes, and model answers.

## Learning objectives

By the end of this lesson, you should be able to:

- Reframe autoregression as supervised learning.
- Construct univariate and multivariate lag matrices.
- Relate an AR model to linear regression on lagged features.

## Setup for this lesson

This cell recreates the data and completed prerequisites from earlier lessons, so this notebook can be run in a fresh kernel.

In [ ]:
# Shared setup from the preceding lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.ar_model import AutoReg
from sklearn.linear_model import LinearRegression

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff'] = df['Log_Passengers'].diff()
series = df['Log_Diff'].dropna()

# The preceding lesson selected p=12 from the seasonal PACF signature.
p = 12
result = AutoReg(series, lags=p).fit()


---
## 📣 Segment 5 — Supervised Learning Framing & Lag Matrix (50 min)

### Why this matters (the big picture)

This is arguably the most important conceptual shift in the course. Once students
understand that **any time series forecasting problem can be converted into a tabular
supervised learning problem**, the full ML toolkit opens up:
- Linear regression
- Random forests / gradient boosting
- Neural networks (MLP)
- RNNs and LSTMs (which process the lag sequence directly)

### Mathematical Structure

Given a series $x_1, x_2, \ldots, x_T$ and lag order $p$, the lag matrix is:

$$X = \begin{pmatrix}
x_1 & x_2 & \cdots & x_p \\
x_2 & x_3 & \cdots & x_{p+1} \\
\vdots & & & \vdots \\
x_{T-p} & x_{T-p+1} & \cdots & x_{T-1}
\end{pmatrix}, \quad
y = \begin{pmatrix} x_{p+1} \\ x_{p+2} \\ \vdots \\ x_T \end{pmatrix}$$

The resulting dataset has $T - p$ rows and $p$ feature columns. Each row is a
'window' of $p$ observations, and the target is the next observation.

Fitting OLS to $(X, y)$ gives the same estimates as `AutoReg` (up to numerical
precision) — the AR model *is* linear regression on lagged features.

### Talking Points
- Walk through the loop in `make_lag_matrix` carefully. Trace through a small
  example (e.g., T=6, p=2) by hand on the board.
- **The 'window view':** Each row is a sliding window of width $p$ that moves
  forward by 1 each step. This is the same operation used in convolutional networks
  and sequence models.
- Segmenyt 7 (the `LinearRegression` comparison) is the payoff: students should
  see that the coefficients are (nearly) identical. This validates both the theory
  and the code.

### ⚠️ Common Mistakes
- **Off-by-one errors in the loop:** Very common. The slice `series[t - n_lags : t]`
  gives exactly `n_lags` values ending at `t-1`. Make sure students understand that
  `t` is the *target* time, not included in the features.
- **Data leakage:** If students include `series[t]` in the features, they've leaked
  the target into the inputs. The model will look perfect but is useless.
- **Forgetting that lag order ≠ window size:** Some students confuse `n_lags` with
  a specific time window in calendar terms. Remind them: 3 lags of monthly data means
  3 months of history, not necessarily 1 quarter (edge cases like skipped periods matter).


### ✅ ANSWER — 5.2 make_lag_matrix (fill-in solution)

In [ ]:
def make_lag_matrix(series, n_lags):
    """
    Convert a 1-D time series into a lag-embedded feature matrix.

    Parameters
    ----------
    series  : array-like, shape (T,)
    n_lags  : int, number of lag features to create

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags)   <- feature matrix
    y : np.ndarray, shape (T - n_lags,)           <- target vector
    """
    series = np.array(series)
    T = len(series)
    X, y = [], []
    for t in range(n_lags, T):
        X.append(series[t - n_lags : t])  # SOLUTION: end index is t
        y.append(series[t])               # SOLUTION: target is series[t]
    return np.array(X), np.array(y)


In [ ]:
X, y = make_lag_matrix(series.values, n_lags=3)
print('Feature matrix X shape:', X.shape)
print('Target vector y shape: ', y.shape)
print()
print('First 5 rows of X:')
print(X[:5])
print()
print('First 5 targets y:')
print(y[:5])


Feature matrix X shape: (140, 3)
Target vector y shape:  (140,)

First 5 rows of X:
[[ 0.05218575  0.1121173  -0.02298952]
 [ 0.1121173  -0.02298952 -0.06402186]
 [-0.02298952 -0.06402186  0.10948423]
 [-0.06402186  0.10948423  0.0919375 ]
 [ 0.10948423  0.0919375   0.        ]]

First 5 targets y:
[-0.06402186  0.10948423  0.0919375   0.         -0.08455739]


### ✅ ANSWER — 5.3 Tidy DataFrame (fill-in solution)

In [ ]:
n_lags = 3
X, y = make_lag_matrix(series.values, n_lags=n_lags)

# SOLUTION: range(1, n_lags+1) gives [1, 2, 3]; add 'target'
col_names = [f'lag_{i}' for i in range(1, n_lags + 1)] + ['target']

lag_df = pd.DataFrame(
    np.column_stack([X, y]),
    columns=col_names
)
print('Shape:', lag_df.shape)
lag_df.head(10)


Shape: (140, 4)


,lag_1,lag_2,lag_3,target
0,0.052186,0.112117,-0.022990,-0.064022
1,0.112117,-0.022990,-0.064022,0.109484
2,-0.022990,-0.064022,0.109484,0.091937
3,-0.064022,0.109484,0.091937,0.000000
4,0.109484,0.091937,0.000000,-0.084557
5,0.091937,0.000000,-0.084557,-0.133531
6,0.000000,-0.084557,-0.133531,-0.134733
7,-0.084557,-0.133531,-0.134733,0.126294
8,-0.133531,-0.134733,0.126294,-0.025752
9,-0.134733,0.126294,-0.025752,0.091350


### ✅ MODEL ANSWER — Written Response 5.3

> **1. Fewer rows:** The lag matrix has $T - p$ rows. We lose the first $p$ observations
> because there aren't enough preceding values to fill all the lag features.
>
> **2. Each row:** Each row represents a single forecasting 'example': the most recent
> $p$ observations (the features) and the next observation (the target we want to predict).
>
> **3. n_lags=12 for monthly data:** This means we use the last 12 months (one full year)
> as features to predict the next month. The model can capture annual seasonality directly
> through the lag-12 feature.


---
## 📣 Segment 6 — Multivariate Lag Matrix (40 min)

### Talking Points
- The multivariate extension is conceptually simple but the implementation is trickier.
  Using `pd.DataFrame.shift()` on the whole DataFrame at once is the cleanest approach.
- Key insight: for $n$ variables and $p$ lags, the feature matrix has $n \times p$ columns.
  For 2 variables and 3 lags: 6 feature columns.
- **Use the dividir y confluir strategy:** assign different groups to explore different
  values of `n_lags` and compare the resulting matrix shapes and the predictive ability.

### ⚠️ Common Mistakes
- **Shifting direction confusion:** `df.shift(k)` shifts values *downward* — i.e.,
  row $t$ gets the value that was at row $t-k$. This is a *lag*, not a lead.
  `df.shift(-k)` would be a lead (future values) — a data leakage trap.
- **Forgetting to drop NaN rows:** After `shift(k)`, the first `k` rows of the shifted
  frame are NaN. After combining $p$ shifts, `dropna()` must be called.
- **Column naming:** Students often lose track of which column corresponds to which
  variable at which lag. Emphasize the naming convention `{variable}_lag{k}`.


In [ ]:
s1 = series.values
np.random.seed(0)
s2 = 0.5 * s1 + np.random.normal(0, 0.02, size=len(s1))
multi_df = pd.DataFrame({'passengers_log_diff': s1, 'indicator': s2})
print('Shape:', multi_df.shape)
multi_df.head()


Shape: (143, 2)


,passengers_log_diff,indicator
0,0.052186,0.061374
1,0.112117,0.064062
2,-0.022990,0.008080
3,-0.064022,0.012807
4,0.109484,0.092093


### ✅ ANSWER — 6.2 make_lag_matrix_multi (fill-in solution)

In [ ]:
def make_lag_matrix_multi(df, n_lags, target_col):
    lagged_frames = []
    for lag in range(1, n_lags + 1):
        shifted = df.shift(lag)                                    # SOLUTION
        shifted.columns = [f'{col}_lag{lag}' for col in df.columns]
        lagged_frames.append(shifted)                              # SOLUTION
    feature_df = pd.concat(lagged_frames, axis=1)
    target = df[target_col]                                        # SOLUTION
    combined = pd.concat([feature_df, target], axis=1).dropna()
    X = combined.drop(columns=[target_col])
    y = combined[target_col]
    return X, y


In [ ]:
X_multi, y_multi = make_lag_matrix_multi(
    df=multi_df, n_lags=3, target_col='passengers_log_diff'
)
print('Feature matrix shape:', X_multi.shape)
print('Target vector shape: ', y_multi.shape)
print('\nColumn names:')
print(list(X_multi.columns))
print()
X_multi.head()


Feature matrix shape: (140, 6)
Target vector shape:  (140,)

Column names:
['passengers_log_diff_lag1', 'indicator_lag1', 'passengers_log_diff_lag2', 'indicator_lag2', 'passengers_log_diff_lag3', 'indicator_lag3']



,passengers_log_diff_lag1,indicator_lag1,passengers_log_diff_lag2,indicator_lag2,passengers_log_diff_lag3,indicator_lag3
3,-0.022990,0.008080,0.112117,0.064062,0.052186,0.061374
4,-0.064022,0.012807,-0.022990,0.008080,0.112117,0.064062
5,0.109484,0.092093,-0.064022,0.012807,-0.022990,0.008080
6,0.091937,0.026423,0.109484,0.092093,-0.064022,0.012807
7,0.000000,0.019002,0.091937,0.026423,0.109484,0.092093


> 🗣️ **Expected column count:** 2 variables × 3 lags = 6 feature columns.
> Column names: `passengers_log_diff_lag1`, `indicator_lag1`, ..., `indicator_lag3`.

### ✅ MODEL ANSWER — Written Response 6.2

> **1. Feature count:** With $n$ variables and $p$ lags, there are $n \times p$ feature
> columns. Here, $2 \times 3 = 6$.
>
> **2. Dropping rows:** The first $p$ rows are dropped because the largest lag
> (`lag_p`) requires $p$ preceding observations. Without enough history, those rows
> have NaN features and can't be used for training.
>
> **3. When is a second variable useful?** When the second variable has predictive
> power for the target beyond what the target's own history provides. Example: past
> fuel prices might help predict airline passenger counts better than passenger
> history alone, since higher prices suppress travel.


---
## 📣 Segment 7 — Synthesis: AR = Linear Regression on Lag Matrix (25 min)

### Talking Points
- This is the payoff of the entire day. When students see that `LinearRegression`
  and `AutoReg` give the same coefficients, it crystallizes the connection between
  classical time series and modern ML.
- Small discrepancies may appear due to: (a) slight differences in how the intercept
  is estimated, (b) treatment of boundary observations. This is normal.
- **Bridge to the rest of the week:** *"Once you have the lag matrix, you can plug it
  into ANY model — random forest, gradient boosting, MLP, even an RNN. That's exactly
  what we'll do over the next three days."*

### ⚠️ Common Mistakes — Final Reflection
- **Not connecting the stability condition to the final answer:** Students should
  explicitly state the $|\phi| < 1$ condition and what it means — not just list it.
- **Treating the lag matrix as optional:** Some students view it as a detail. Push
  back: it's the foundation for everything in Days 3–5.


In [ ]:
from sklearn.linear_model import LinearRegression

X_lr, y_lr = make_lag_matrix(series.values, n_lags=p)

lr = LinearRegression(fit_intercept=True)
lr.fit(X_lr, y_lr)

print('LinearRegression coefficients (lag_1, ..., lag_p):')
print(lr.coef_)
print('Intercept:', lr.intercept_)
print()
print('AutoReg coefficients (intercept, lag_1, ..., lag_p):')
print(result.params.values)


LinearRegression coefficients (lag_1, ..., lag_p):
[ 0.59971786 -0.20392684 -0.30684049 -0.22537999 -0.35361729 -0.25933241
 -0.29079106 -0.23012178 -0.31721135 -0.25703053 -0.29749863 -0.22129466]
Intercept: 0.03408404623928041

AutoReg coefficients (intercept, lag_1, ..., lag_p):
[ 0.03408405 -0.22129466 -0.29749863 -0.25703053 -0.31721135 -0.23012178
 -0.29079106 -0.25933241 -0.35361729 -0.22537999 -0.30684049 -0.20392684
  0.59971786]


> 🗣️ **Expected output:** The coefficients should be nearly identical (within numerical
> precision). Small differences at the boundary are expected.

### ✅ MODEL ANSWER — Final Written Reflection

> An AR($p$) model predicts the current value of a time series as a weighted sum of the
> previous $p$ values plus random noise: $x_t = \phi_1 x_{t-1} + \cdots + \phi_p x_{t-p}
> + \varepsilon_t$. The coefficients $\phi_i$ quantify how strongly each past value
> influences the present.
>
> The stability condition $|\phi| < 1$ (for AR(1)) guarantees that the process is
> stationary — it has a finite, constant mean and variance. Without this condition, the
> model can produce non-stationary or explosive forecasts that are meaningless in practice.
>
> We chose the order $p$ by reading the PACF: the number of significant lags outside
> the 95% confidence band tells us how many direct lag terms to include. For the airline
> data, the strong spike at lag 12 suggested $p = 12$.
>
> By stacking lagged values as columns, we convert the time series into a tabular
> dataset where each row is a training example. This lag-embedded feature matrix allows
> any supervised learning algorithm to be applied to time series forecasting.
>
> This re-framing is powerful because it unlocks tree-based models (which capture
> non-linear lag interactions), MLPs (which can learn complex feature combinations),
> and RNNs/LSTMs (which process the lag sequence with memory — the topic of Day 5).


---
## 🔚 End-of-Day Wrap-Up Notes

### What students should walk away knowing
1. The AR($p$) model uses the last $p$ values to predict the next one.
2. The stability condition $|\phi| < 1$ (AR(1)) ensures stationarity.
3. PACF cuts off at lag $p$ for AR($p$) — use it to select model order.
4. A lag-embedded feature matrix turns time series forecasting into supervised learning.
5. AR($p$) = linear regression on a lag matrix — any ML model can replace the linear fit.

### Preview for Day 3
> Tomorrow we'll implement and evaluate **multiple forecasting models** side by side:
> a naive baseline, the AR model from today, and tree-based models (Random Forest,
> Gradient Boosting). We'll introduce **time-aware train/test splits** and compare
> models using RMSE and MAE.

### Frequently Confused Pairs

| Students confuse... | Clarification |
|---|---|
| AR order $p$ vs. lag window in months | $p=12$ means 12 lag features, which for monthly data = 12 months of history |
| $\phi$ coefficient vs. autocorrelation | Equal only for AR(1); differ for higher orders |
| PACF cut-off vs. ACF cut-off | PACF → AR order; ACF → MA order |
| `df.shift(1)` vs. `df.shift(-1)` | `shift(1)` = lag (past); `shift(-1)` = lead (future, data leakage!) |
| AR model fitting on raw data | Must use stationary (transformed) series for valid inference |
